# Project introduction — multimodal stock intelligence

This notebook introduces the project before the hands-on real-data notebooks.

It answers three questions:

1. **What problem are we solving?**
2. **Why do we need multiple modalities?**
3. **How do tabular, image, text, and KG data become embeddings for a fusion Transformer?**

This notebook is intentionally **conceptual only**. It does not show backtest numbers, return curves, or performance claims.

## Why this project exists

Traditional stock prediction pipelines often rely on tabular price and volume features alone. But a human analyst usually combines several views of the same stock and date window:

- numeric time-series behavior;
- chart structure and price patterns;
- text or narrative context available by that date;
- peer, sector, and event relationships.

This project asks whether a Transformer can combine those views into one aligned representation.

## Conceptual workflow: raw modalities to embeddings

The diagram below explains the end-to-end workflow conceptually. It is not a result chart.

In [ ]:
# Conceptual diagram 1: raw multimodal inputs -> aligned embedding artifact.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def box(ax, xy, text, width=2.6, height=0.72, fc='#eef6ff', ec='#4a90e2', fontsize=10):
    patch = FancyBboxPatch(
        xy, width, height,
        boxstyle='round,pad=0.05,rounding_size=0.08',
        linewidth=1.4, edgecolor=ec, facecolor=fc
    )
    ax.add_patch(patch)
    ax.text(xy[0] + width/2, xy[1] + height/2, text, ha='center', va='center', fontsize=fontsize)

def arrow(ax, start, end):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='->', mutation_scale=14, linewidth=1.2, color='#555555'))

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis('off')

ax.text(0.4, 6.5, 'Raw multimodal inputs', fontsize=14, weight='bold')
box(ax, (0.4, 5.35), 'Tabular market data\nOHLCV, returns, volatility')
box(ax, (0.4, 4.25), 'Chart images\nCandlestick PNG windows', fc='#f5efff', ec='#8e5cc2')
box(ax, (0.4, 3.15), 'Text records\nevent_date <= sample date', fc='#eefaf1', ec='#4c9f70')
box(ax, (0.4, 2.05), 'KG context\nsector, peer, event graph', fc='#fff5e8', ec='#d68a2d')

ax.text(4.0, 6.5, 'Preprocess and align', fontsize=14, weight='bold')
box(ax, (4.0, 5.0), 'Feature engineering\nfuture outperformance label', width=3.0)
box(ax, (4.0, 3.9), 'Chart generation\nas-of-date window', width=3.0, fc='#f5efff', ec='#8e5cc2')
box(ax, (4.0, 2.8), 'Text/KG cutoff\nno future leakage', width=3.0, fc='#eefaf1', ec='#4c9f70')
box(ax, (4.0, 1.7), 'Alignment key\nstock_id + end_date', width=3.0, fc='#fff5e8', ec='#d68a2d')

ax.text(8.2, 6.5, 'Embedding artifact', fontsize=14, weight='bold')
box(ax, (8.25, 4.95), 'tabular_tokens', width=2.8)
box(ax, (8.25, 4.05), 'image_tokens', width=2.8, fc='#f5efff', ec='#8e5cc2')
box(ax, (8.25, 3.15), 'text_tokens', width=2.8, fc='#eefaf1', ec='#4c9f70')
box(ax, (8.25, 2.25), 'kg_tokens', width=2.8, fc='#fff5e8', ec='#d68a2d')
box(ax, (8.25, 1.25), 'y + metadata', width=2.8, fc='#f2f2f2', ec='#777777')

for y in [5.7, 4.6, 3.5, 2.4]:
    arrow(ax, (3.05, y), (3.95, y - 0.2))
for y in [5.35, 4.25, 3.15, 2.05]:
    arrow(ax, (7.05, y - 0.05), (8.2, y - 0.05))

ax.text(0.4, 0.45, 'Takeaway: heterogeneous raw data becomes synchronized vectors for the same stock/date sample.', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

## What each modality contributes

- **Tabular** captures returns, volatility, volume behavior, and benchmark-relative strength.
- **Image** captures chart geometry and visual price structure.
- **Text** captures narrative/context available on or before the sample date.
- **Knowledge graph** captures sector, peer, and event relationships.

The important design rule is that every modality must describe the same `(stock_id, end_date)` sample, without leaking future information.

## Conceptual fusion architecture

The second diagram explains how modality-specific vectors are combined in embedding space. This is an architecture explanation only, not a performance result.

In [ ]:
# Conceptual diagram 2: modality embeddings -> fusion Transformer -> prediction.
fig, ax = plt.subplots(figsize=(14, 6.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6.5)
ax.axis('off')

ax.text(0.4, 6.0, 'Aligned modality embeddings', fontsize=14, weight='bold')
box(ax, (0.5, 4.9), 'tabular embedding', width=2.4)
box(ax, (0.5, 3.9), 'image embedding', width=2.4, fc='#f5efff', ec='#8e5cc2')
box(ax, (0.5, 2.9), 'text embedding', width=2.4, fc='#eefaf1', ec='#4c9f70')
box(ax, (0.5, 1.9), 'kg embedding', width=2.4, fc='#fff5e8', ec='#d68a2d')

ax.text(4.35, 6.0, 'Fusion in shared embedding space', fontsize=14, weight='bold')
box(ax, (4.1, 4.75), 'Project to common dimension', width=3.2, fc='#eef6ff')
box(ax, (4.1, 3.75), 'Concatenate modality tokens', width=3.2, fc='#eef6ff')
box(ax, (4.1, 2.75), 'Self-attention across modalities', width=3.2, fc='#eef6ff')
box(ax, (4.1, 1.75), 'Fused representation', width=3.2, fc='#eef6ff')

ax.text(8.65, 6.0, 'Outputs and interpretation', fontsize=14, weight='bold')
box(ax, (8.7, 4.75), 'Probability of\noutperformance vs Nifty', width=2.8, fc='#f2f2f2', ec='#777777')
box(ax, (8.7, 3.45), 'Cross-modal reasoning\ntrend + chart + text + KG', width=2.8, fc='#f2f2f2', ec='#777777')
box(ax, (8.7, 2.15), 'Ablation comparison\nwhich modality changes behavior?', width=2.8, fc='#f2f2f2', ec='#777777')

for y in [5.25, 4.25, 3.25, 2.25]:
    arrow(ax, (2.95, y), (4.05, 4.95))
arrow(ax, (7.35, 3.1), (8.65, 5.1))
arrow(ax, (7.35, 3.1), (8.65, 3.8))
arrow(ax, (7.35, 3.1), (8.65, 2.5))

ax.text(0.5, 0.55, 'Takeaway: fusion lets the model mix signals that a single modality would see in isolation.', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

## What the notebooks will show next

The rest of the Colab notebook sequence makes this story concrete:

1. download real OHLCV and benchmark data;
2. engineer features and labels;
3. build aligned multimodal tensors;
4. run ablations;
5. generate actual-data visualizations from the embeddings and outputs.

## Important honesty note

This introduction notebook is for **conceptual framing**. It deliberately avoids showing any backtest or numeric performance diagram.

For evidence, use the later notebooks that generate real artifacts from actual data:

- real candlestick charts;
- actual token tensors;
- actual ablation metrics;
- actual embedding projections.

A full portfolio backtest is a future notebook/script and should not be claimed yet.

## Next notebook

Proceed to: **`00_setup_and_data_download.ipynb`**